In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

In [21]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [22]:
import markdown
from xhtml2pdf import pisa

In [23]:
from pathlib import Path
import pandas as pd

carpeta_erp = Path("Ventas Diarias ERP")

lista_cabecera = []
lista_detalle = []

for archivo in carpeta_erp.glob("Ventas_ERP_*.xlsx"):
    # Nombres exactos como figuran abajo en la imagen de Excel
    df_cab = pd.read_excel(archivo, sheet_name='Nivel Transaccion')
    df_det = pd.read_excel(archivo, sheet_name='Nivel Detalle de Item')
    
    lista_cabecera.append(df_cab)
    lista_detalle.append(df_det)

df_cabecera = pd.concat(lista_cabecera, ignore_index=True)
df_detalle = pd.concat(lista_detalle, ignore_index=True)

In [24]:


# Cambiar el directorio de trabajo a la carpeta real del proyecto
#os.chdir(r"C:\Users\matia\OneDrive\Desktop\data\Github\Proyecto automatizacion")

# Ahora sí podés cargar el Excel sin problemas
#ruta_consolidado = "Ventas_Diarias_Consolidados.xlsx"

#df_detalle = pd.read_excel(ruta_consolidado, sheet_name='Fact_Ventas_Detalle')
#df_cabecera = pd.read_excel(ruta_consolidado, sheet_name='Fact_Ventas_Cabecera')

In [25]:
df_completo = pd.merge(df_detalle, df_cabecera, on= 'ID Transaccion')

In [26]:
df_completo

,ID Linea,ID Transaccion,Nro Linea,SKU,Producto,Categoria,Cantidad,Precio Unitario,Costo Unitario,Descuento Linea,...,Canal Venta,ID Cliente,Metodo Pago,Vendedor,Subtotal,Descuento Total,Impuestos (IVA 21%),Total Cobrado,Costo Total ERP,Margen Bruto
0,1,TXN-20260727-001,1,PROD-102,Monitor 27 UHD,Tecnología,3,280000,190000,84000,...,Tienda Física,CLI-1520,Tarjeta Crédito,VEN-102 (María Gómez),NaN,NaN,NaN,NaN,NaN,NaN
1,2,TXN-20260727-002,1,PROD-104,Mouse Inalámbrico,Accesorios,3,25000,11000,7500,...,E-Commerce,CLI-4611,Efectivo,SISTEMA-WEB,NaN,NaN,NaN,NaN,NaN,NaN
2,3,TXN-20260727-003,1,PROD-105,Auriculares Bluetooth,Audio,1,65000,35000,0,...,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López),NaN,NaN,NaN,NaN,NaN,NaN
3,4,TXN-20260727-003,2,PROD-106,Silla Ergonómica Oficina,Muebles,1,185000,115000,0,...,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López),NaN,NaN,NaN,NaN,NaN,NaN
4,5,TXN-20260727-003,3,PROD-107,Escritorio Ajustable,Muebles,1,240000,150000,12000,...,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López),NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959,75,TXN-20260807-030,3,PROD-106,Silla Ergonómica Oficina,Muebles,1,185000,115000,0,...,Tienda Física,CLI-4685,Tarjeta Crédito,VEN-103 (Carlos Ruiz),NaN,NaN,NaN,NaN,NaN,NaN
960,76,TXN-20260807-031,1,PROD-102,Monitor 27 UHD,Tecnología,2,280000,190000,28000,...,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB,NaN,NaN,NaN,NaN,NaN,NaN
961,77,TXN-20260807-031,2,PROD-103,Teclado Mecánico RGB,Accesorios,1,45000,22000,2250,...,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB,NaN,NaN,NaN,NaN,NaN,NaN
962,78,TXN-20260807-031,3,PROD-104,Mouse Inalámbrico,Accesorios,2,25000,11000,0,...,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# 1. Calcular el 'Total' exacto de la línea
df_completo['Total'] = (df_completo['Cantidad'] * df_completo['Precio Unitario']) - df_completo['Descuento Linea']

# 2. Definir el orden exacto de las columnas que necesitás
columnas_deseadas = [
    'ID Linea', 'ID Transaccion', 'Nro Linea', 'SKU', 'Producto', 
    'Categoria', 'Cantidad', 'Precio Unitario', 'Costo Unitario', 
    'Descuento Linea', 'Total', 'Fecha Hora', 'Sucursal', 
    'Canal Venta', 'ID Cliente', 'Metodo Pago', 'Vendedor'
]

# 3. Filtrar y reordenar el DataFrame
df_completo = df_completo[columnas_deseadas]

In [28]:
df_completo

,ID Linea,ID Transaccion,Nro Linea,SKU,Producto,Categoria,Cantidad,Precio Unitario,Costo Unitario,Descuento Linea,Total,Fecha Hora,Sucursal,Canal Venta,ID Cliente,Metodo Pago,Vendedor
0,1,TXN-20260727-001,1,PROD-102,Monitor 27 UHD,Tecnología,3,280000,190000,84000,756000,2026-07-27 09:47:17,SUC-001 Centro,Tienda Física,CLI-1520,Tarjeta Crédito,VEN-102 (María Gómez)
1,2,TXN-20260727-002,1,PROD-104,Mouse Inalámbrico,Accesorios,3,25000,11000,7500,67500,2026-07-27 10:13:14,SUC-003 E-Commerce,E-Commerce,CLI-4611,Efectivo,SISTEMA-WEB
2,3,TXN-20260727-003,1,PROD-105,Auriculares Bluetooth,Audio,1,65000,35000,0,65000,2026-07-27 18:17:51,SUC-001 Centro,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López)
3,4,TXN-20260727-003,2,PROD-106,Silla Ergonómica Oficina,Muebles,1,185000,115000,0,185000,2026-07-27 18:17:51,SUC-001 Centro,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López)
4,5,TXN-20260727-003,3,PROD-107,Escritorio Ajustable,Muebles,1,240000,150000,12000,228000,2026-07-27 18:17:51,SUC-001 Centro,Tienda Física,CLI-6635,Transferencia,VEN-104 (Ana López)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959,75,TXN-20260807-030,3,PROD-106,Silla Ergonómica Oficina,Muebles,1,185000,115000,0,185000,2026-08-07 18:37:08,SUC-002 Norte,Tienda Física,CLI-4685,Tarjeta Crédito,VEN-103 (Carlos Ruiz)
960,76,TXN-20260807-031,1,PROD-102,Monitor 27 UHD,Tecnología,2,280000,190000,28000,532000,2026-08-07 11:38:56,SUC-003 E-Commerce,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB
961,77,TXN-20260807-031,2,PROD-103,Teclado Mecánico RGB,Accesorios,1,45000,22000,2250,42750,2026-08-07 11:38:56,SUC-003 E-Commerce,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB
962,78,TXN-20260807-031,3,PROD-104,Mouse Inalámbrico,Accesorios,2,25000,11000,0,50000,2026-08-07 11:38:56,SUC-003 E-Commerce,E-Commerce,CLI-2499,Tarjeta Débito,SISTEMA-WEB


In [29]:
df_completo.info()

<class 'pandas.DataFrame'>
RangeIndex: 964 entries, 0 to 963
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ID Linea         964 non-null    int64         
 1   ID Transaccion   964 non-null    str           
 2   Nro Linea        964 non-null    int64         
 3   SKU              964 non-null    str           
 4   Producto         964 non-null    str           
 5   Categoria        964 non-null    str           
 6   Cantidad         964 non-null    int64         
 7   Precio Unitario  964 non-null    int64         
 8   Costo Unitario   964 non-null    int64         
 9   Descuento Linea  964 non-null    int64         
 10  Total            964 non-null    int64         
 11  Fecha Hora       964 non-null    datetime64[us]
 12  Sucursal         964 non-null    str           
 13  Canal Venta      964 non-null    str           
 14  ID Cliente       964 non-null    str           
 15  

LIMPIEZA

In [30]:
df_completo['Total'] = df_completo['Total'].astype(float)
df_completo['Precio Unitario'] = df_completo['Precio Unitario'].astype(float)
df_completo['Costo Unitario'] = df_completo['Costo Unitario'].astype(float)
df_completo['Descuento Linea'] = df_completo['Descuento Linea'].astype(float)

In [31]:
df_completo['Fecha'] =df_completo['Fecha Hora'].dt.date
df_completo['Hora'] =df_completo['Fecha Hora'].dt.time

In [32]:
df_completo['Fecha'].head()

0    2026-07-27
1    2026-07-27
2    2026-07-27
3    2026-07-27
4    2026-07-27
Name: Fecha, dtype: object

ANALISIS DE VENTAS

In [33]:
Total_vendido_por_producto = df_completo.groupby('Producto')['Total'].sum().reset_index()

In [34]:
Total_vendido_por_producto = Total_vendido_por_producto.sort_values('Total', ascending=False)
Total_vendido_por_producto

,Producto,Total
4,Notebook Pro 15,185427500.0
2,Monitor 27 UHD,84644000.0
1,Escritorio Ajustable,56112000.0
5,Silla Ergonómica Oficina,44871750.0
7,Webcam 4K Ultra,16173750.0
0,Auriculares Bluetooth,14118000.0
6,Teclado Mecánico RGB,9441000.0
3,Mouse Inalámbrico,6712500.0


In [35]:
fig, ax = plt.subplots(figsize=(10, 9))

bars = ax.bar( Total_vendido_por_producto['Producto'], Total_vendido_por_producto['Total'], color="#2c3e50")


ax.set_title('Total vendido por producto',fontsize=12, fontweight='bold', pad=15)
#ax.set_xlabel('Productos')
ax.set_ylabel('Facturación ($)')

# Formatear eje Y a Millones ($M) en lugar de notación científica (1e7)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${x/1e6:.0f}M'))

# Bucle para poner etiquetas sobre cada punto
for x, y in zip(Total_vendido_por_producto['Producto'], Total_vendido_por_producto['Total']):
    ax.annotate(
        f'${y/1e6:.1f}M',          # 2 decimales ($46.92M) o .1f para uno ($46.9M)
        (x, y),                     # Coordenadas del punto
        textcoords="offset points", 
        xytext=(2, 10),              # Desplazamiento vertical para que no solape la línea
        ha='center',                # Alineación horizontal al centro
        fontsize=8,
        fontweight='bold',
        color='#2c3e50'
    )

# Sacar eje Y y bordes superior/izquierdo/derecho
#ax.yaxis.set_visible(False)
#for spine in ['top', 'right', 'left']:
    #ax.spines[spine].set_visible(False) 


 #  Ajustar límites en Y para dar margen vertical y evitar recortes de texto
y_min = Total_vendido_por_producto['Total'].min()
y_max = Total_vendido_por_producto['Total'].max()
ax.set_ylim(bottom=y_min * 0.0, top=y_max * 1.12)

ax.tick_params(axis='x', labelrotation=32)

plt.tight_layout()
# Guardar como PNG de alta resolución para el PDF
plt.savefig('images/total_vendido_por_producto', dpi=300, bbox_inches='tight')
plt.close() # Cierra la figura en memoria

plt.show()

In [36]:
Total_vendido_por_canal = df_completo.groupby('Canal Venta')['Total'].sum().reset_index()

In [37]:
Total_vendido_por_canal = Total_vendido_por_canal.sort_values('Total', ascending=False)
Total_vendido_por_canal

,Canal Venta,Total
2,Venta Telefónica,149244750.0
0,E-Commerce,148847500.0
1,Tienda Física,119408250.0


In [38]:
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar( Total_vendido_por_canal['Canal Venta'], Total_vendido_por_canal['Total'], color='#2c3e50')

ax.set_title('Total vendido por canal de venta',fontsize=12, fontweight='bold', pad=15)
#ax.set_xlabel('Canal de venta')
ax.set_ylabel('Facturación ($)')

# Formatear eje Y a Millones ($M) en lugar de notación científica (1e7)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${x/1e6:.0f}M'))

# Bucle para poner etiquetas sobre cada punto
for x, y in zip(Total_vendido_por_canal['Canal Venta'], Total_vendido_por_canal['Total']):
    ax.annotate(
        f'${y/1e6:.1f}M',          # 2 decimales ($46.92M) o .1f para uno ($46.9M)
        (x, y),                     # Coordenadas del punto
        textcoords="offset points", 
        xytext=(2, 10),              # Desplazamiento vertical para que no solape la línea
        ha='center',                # Alineación horizontal al centro
        fontsize=8,
        fontweight='bold',
        color='#2c3e50'
    )

# Sacar eje Y y bordes superior/izquierdo/derecho
#ax.yaxis.set_visible(False)
#for spine in ['top', 'right', 'left']:
    #ax.spines[spine].set_visible(False) 


 #  Ajustar límites en Y para dar margen vertical y evitar recortes de texto
y_min = Total_vendido_por_producto['Total'].min()
y_max = Total_vendido_por_producto['Total'].max()
ax.set_ylim(bottom=y_min * 0.0, top=y_max * 1.12)

ax.tick_params(axis='x', labelrotation=35)

plt.tight_layout()

# Guardar como PNG de alta resolución para el PDF
plt.savefig('images/venta_por_canal_de_venta.png', dpi=300, bbox_inches='tight')
plt.close() # Cierra la figura en memoria

plt.show()

In [39]:
Cantidad_vendida = df_completo.groupby('Producto')['Cantidad'].sum().reset_index()

In [40]:
Cantidad_vendida = Cantidad_vendida.sort_values('Cantidad', ascending=False)
Cantidad_vendida

,Producto,Cantidad
2,Monitor 27 UHD,315
3,Mouse Inalámbrico,281
5,Silla Ergonómica Oficina,252
1,Escritorio Ajustable,241
4,Notebook Pro 15,227
0,Auriculares Bluetooth,226
7,Webcam 4K Ultra,224
6,Teclado Mecánico RGB,218


In [41]:
fig, ax = plt.subplots(figsize=(10, 9))

bars = ax.barh(Cantidad_vendida['Producto'], Cantidad_vendida['Cantidad'], color='#2c3e50')

ax.set_title('Unidades Vendidas por Producto',fontsize=12, fontweight='bold', pad=15)
#ax.set_xlabel('Cantidad de Unidades')
#ax.set_ylabel('Producto')

# 4. Invertir eje Y para mostrar el mayor volumen arriba
ax.invert_yaxis()

# 6. Ampliar el límite horizontal un 10% para que la etiqueta '176' no toque el borde
ax.set_xlim(right=Cantidad_vendida['Cantidad'].max() * 1.10)

ax.tick_params(axis='x', labelrotation=0)
ax.bar_label(bars)
plt.tight_layout()

# Guardar como PNG de alta resolución para el PDF
plt.savefig('images/ventas_por_producto.png', dpi=300, bbox_inches='tight')
plt.close() # Cierra la figura en memoria


plt.show()

In [42]:
Facturacion_diaria = df_completo.groupby('Fecha')['Total'].sum().reset_index()
Facturacion_diaria


,Fecha,Total
0,2026-07-27,25867250.0
1,2026-07-28,40055500.0
2,2026-07-29,42682250.0
3,2026-07-30,33343250.0
4,2026-07-31,24447500.0
5,2026-08-01,46916250.0
6,2026-08-02,36266250.0
7,2026-08-03,30951500.0
8,2026-08-04,19981000.0
9,2026-08-05,43335500.0


In [43]:
Fecha = Facturacion_diaria['Fecha']
Total = Facturacion_diaria['Total']

In [44]:
fix, ax= plt.subplots(figsize=(12, 6))
ax.plot(Facturacion_diaria['Fecha'], Facturacion_diaria['Total'], linestyle='-',  marker='o', markersize=6, color='#2c3e50')
ax.set_title('Facturacion diaria',fontsize=12, fontweight='bold', pad=15)
#ax.set_xlabel('Dias')
#ax.set_ylabel('Facturación ($)')

# Formatear eje Y a Millones ($M) en lugar de notación científica (1e7)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${x/1e6:.0f}M'))

# Bucle para poner etiquetas sobre cada punto
for x, y in zip(Facturacion_diaria['Fecha'], Facturacion_diaria['Total']):
    ax.annotate(
        f'${y/1e6:.1f}M',          # 2 decimales ($46.92M) o .1f para uno ($46.9M)
        (x, y),                     # Coordenadas del punto
        textcoords="offset points", 
        xytext=(2, 12),              # Desplazamiento vertical para que no solape la línea
        ha='center',                # Alineación horizontal al centro
        fontsize=8,
        fontweight='bold',
        color='#2c3e50'
    )

# 6. Ajustar límites en Y para dar margen vertical y evitar recortes de texto
y_min = Facturacion_diaria['Total'].min()
y_max = Facturacion_diaria['Total'].max()
ax.set_ylim(bottom=y_min * 0.85, top=y_max * 1.12)

ax.tick_params(axis='x', labelrotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()

# Guardar como PNG de alta resolución para el PDF
plt.savefig('images/evolucion_diaria.png', dpi=300, bbox_inches='tight')
plt.close() # Cierra la figura en memoria

plt.show()

SE COMBIERTEN LOS DATOS PRSESADOS EN TEXTO O DICCIONARIO PARA PODER ENVIÁRSELOS A LA API


In [45]:
Resumen_ventas_procuctos = Total_vendido_por_producto.to_string(index=False)
Resumen_ventas_canal = Total_vendido_por_canal.to_string(index=False)
Resumen_cantidad_por_canal = Cantidad_vendida.to_string(index=False)
Resumen_ventas_diarias = Facturacion_diaria.to_string(index=False)

In [46]:
prompt_datos = f"""
Eres un analista financiero y de negocios senior. Analiza los siguientes datos de ventas consolidadas:

1. Ventas por Producto:
{Resumen_ventas_procuctos}

2. Ventas por Canal de Venta:
{Resumen_ventas_canal}

3. Cantidad por Canal de Venta:
{Resumen_cantidad_por_canal}

4. Evolución Diaria de Ventas:
{Resumen_ventas_diarias}

Reglas estrictas de respuesta:
- Empieza DIRECTAMENTE en el encabezado "## 1. Resumen Ejecutivo" (NO incluyas títulos principales ni metadatos de Para/De/Fecha al inicio).
- Escribe ÚNICAMENTE texto analítico y viñetas estándar en Markdown.
- Queda ESTRICTAMENTE PROHIBIDO usar caracteres especiales o código ASCII para crear barras de progreso, gráficos de texto o bloques (ej. NO uses ■, █, |||).
- Redacta con las siguientes secciones:
  1. Resumen Ejecutivo
  2. Hallazgos Clave: Portafolio de Productos
  3. Desempeño de los Canales de Venta
  4. Evolución Diaria de Ventas
  5. Conclusiones y Recomendaciones Estratégicas
"""


CONECTAMOS A LA API DE GEMINI

In [47]:

load_dotenv()


# Inicializamos el cliente con tu API Key
client =genai.Client()

# Configurar la temperatura a 0.1 para un análisis preciso
config_reporte = types.GenerateContentConfig(
    temperature=0.1,
    top_p=0.95
)

# Generamos el informe
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=prompt_datos,
    config=config_reporte
)

texto_informe = response.text
#print(texto_informe)

In [48]:
import re
import markdown
from datetime import datetime
import locale
from xhtml2pdf import pisa

# Configurar idioma en español
try:
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    try:
        locale.setlocale(locale.LC_TIME, 'spanish')
    except locale.Error:
        pass

# Generar fecha actual dinámicamente
fecha_actual = datetime.now().strftime("%d de %B de %Y").capitalize()

# CALCULAR PERÍODO DINÁMICO (Usa el DataFrame con las fechas de ventas)
try:
    fecha_min = Facturacion_diaria['Fecha'].min().strftime("%d/%m/%Y")
    fecha_max = Facturacion_diaria['Fecha'].max().strftime("%d/%m/%Y")
    periodo_analizado = f"{fecha_min} al {fecha_max}"
except NameError:
    # Backup manual por si la variable no estuviera definida en este scope
    periodo_analizado = "27/07/2026 al 03/08/2026"

# 1. Limpieza estricta del texto Markdown generado por el LLM
texto_informe_limpio = re.sub(
    r'^.*?(?=\#\#?\s*1\.\s*Resumen Executive|\*\*1\.\s*Resumen Executive|1\.\s*Resumen Executive|\#\#?\s*1\.\s*RESUMEN EJECUTIVO|\*\*1\.\s*RESUMEN EJECUTIVO|1\.\s*RESUMEN EJECUTIVO)', 
    '', 
    texto_informe, 
    flags=re.DOTALL | re.IGNORECASE
)

texto_informe_limpio = re.sub(r'[■█▌▐░▒▓]', '', texto_informe_limpio)

html_cuerpo = markdown.markdown(texto_informe_limpio, extensions=['tables'])

colgroup_html = """
<colgroup>
    <col style="width: 22%;">
    <col style="width: 18%;">
    <col style="width: 14%;">
    <col style="width: 12%;">
    <col style="width: 14%;">
    <col style="width: 20%;">
</colgroup>
"""
html_cuerpo = re.sub(r'(<table>)', r'\1' + colgroup_html, html_cuerpo)

# 2. INSERCIÓN ORDENADA DE GRÁFICOS POR SECCIÓN
if re.search(r'<h[23][^>]*>.*?CANALES.*?</h[23]>', html_cuerpo, flags=re.IGNORECASE):
    html_cuerpo = re.sub(
        r'(<h[23][^>]*>.*?CANALES.*?</h[23]>)',
        r'<table class="tabla-graficos-doble">'
        r'<tr>'
        r'<td><img src="images/total_vendido_por_producto.png" alt="Monto Facturado por Producto"></td>'
        r'<td><img src="images/ventas_por_producto.png" alt="Unidades Vendidas por Producto"></td>'
        r'</tr>'
        r'</table>'
        r'<div class="salto-pagina"></div>\1',
        html_cuerpo,
        flags=re.IGNORECASE,
        count=1
    )

if re.search(r'<h[23][^>]*>.*?EVOLUCIÓN.*?</h[23]>', html_cuerpo, flags=re.IGNORECASE):
    html_cuerpo = re.sub(
        r'(<h[23][^>]*>.*?EVOLUCIÓN.*?</h[23]>)',
        r'<div class="contenedor-grafico-inline"><img src="images/venta_por_canal_de_venta.png" alt="Ventas por Canal"></div><div class="salto-pagina"></div>\1',
        html_cuerpo,
        flags=re.IGNORECASE,
        count=1
    )

if re.search(r'<h[23][^>]*>.*?CONCLUSIONES.*?</h[23]>', html_cuerpo, flags=re.IGNORECASE):
    html_cuerpo = re.sub(
        r'(<h[23][^>]*>.*?CONCLUSIONES.*?</h[23]>)',
        r'<div class="contenedor-grafico-inline"><img src="images/evolucion_diaria.png" alt="Evolución Diaria de Ventas"></div><div class="salto-pagina"></div>\1',
        html_cuerpo,
        flags=re.IGNORECASE,
        count=1
    )

# 3. Plantilla HTML y Estilos CSS
html_completo = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <style>
        @page {{
            size: a4;
            margin: 1.2cm;
        }}
        body {{
            font-family: Helvetica, Arial, sans-serif;
            color: #2c3e50;
            font-size: 8.5pt;
            line-height: 1.3;
        }}
        
        .header {{
            border-bottom: 2px solid #2980b9;
            padding-bottom: 4px;
            margin-bottom: 8px;
        }}
        h1 {{
            color: #2c3e50;
            font-size: 14pt;
            margin: 0;
        }}
        .date {{
            color: #7f8c8d;
            font-size: 7.5pt;
            margin-top: 2px;
        }}
        
        .meta-tabla {{
            width: 100%;
            border-collapse: collapse;
            margin-bottom: 6px;
            margin-top: 4px;
        }}
        .meta-tabla td {{
            border: none !important;
            padding: 1px 0;
            font-size: 7.5pt;
            background-color: transparent !important;
        }}
        .meta-label {{
            font-weight: bold;
            color: #2980b9;
            width: 10%;
        }}
        .meta-val {{
            color: #2c3e50;
            width: 40%;
        }}
        .header-divider {{
            border: none;
            border-top: 1px solid #cbd5e1;
            margin-bottom: 8px;
        }}

        .salto-pagina {{
            page-break-before: always;
        }}

        h2, h3 {{
            color: #2980b9;
            font-size: 10.5pt;
            margin-top: 8px;
            margin-bottom: 6px;
            border-bottom: 1px solid #eee;
            padding-bottom: 2px;
            page-break-after: avoid;
        }}
        
        p, li {{
            text-align: justify;
            margin-bottom: 5px;
        }}

        ul, ol {{
            margin-left: 15px;
            margin-bottom: 6px;
        }}

        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 6px;
            margin-bottom: 8px;
            page-break-inside: avoid;
        }}
        tr {{
            page-break-inside: avoid;
        }}
        th, td {{
            border: 0.5pt solid #cbd5e1;
            padding: 3px 3px;
            font-size: 6.5pt;
            vertical-align: middle;
            text-align: left;
        }}
        th {{
            background-color: #2980b9;
            color: #ffffff;
            font-weight: bold;
        }}
        tr:nth-child(even) {{
            background-color: #f8fafc;
        }}
        
        .contenedor-grafico-inline {{
            text-align: center;
            margin-top: 10px;
            margin-bottom: 6px;
            page-break-inside: avoid;
        }}
        .contenedor-grafico-inline img {{
            width: 100%;
            max-width: 410px;
            height: auto;
        }}

        .tabla-graficos-doble {{
            width: 100%;
            border: none !important;
            margin-top: 8px;
            margin-bottom: 6px;
            page-break-inside: avoid;
        }}
        .tabla-graficos-doble td {{
            width: 50%;
            border: none !important;
            padding: 0 4px;
            text-align: center;
            background-color: transparent !important;
        }}
        .tabla-graficos-doble img {{
            width: 100%;
            height: auto;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Informe Ejecutivo de Ventas Diarias</h1>
        <div class="date">Proyecto Automatización — Generado automáticamente</div>
    </div>

    <table class="meta-tabla">
        <tr>
            <td class="meta-label">Para:</td>
            <td class="meta-val">Dirección Ejecutiva y Junta Directiva</td>
            <td class="meta-label">Fecha:</td>
            <td class="meta-val">{fecha_actual}</td>
        </tr>
        <tr>
            <td class="meta-label">De:</td>
            <td class="meta-val">Analista Financiero y de Negocios Senior</td>
            <td class="meta-label">Período:</td>
            <td class="meta-val">{periodo_analizado}</td>
        </tr>
        <tr>
            <td class="meta-label">Asunto:</td>
            <td colspan="3" class="meta-val">Análisis Consolidado de Ventas y Rendimiento Comercial</td>
        </tr>
    </table>
    <hr class="header-divider">

    <div class="cuerpo-informe">
        {html_cuerpo}
    </div>
</body>
</html>
"""

# 4. Generación del PDF
ruta_pdf = "reports/Informe_Ejecutivo_Ventas.pdf"

with open(ruta_pdf, "wb") as pdf_file:
    pisa_status = pisa.CreatePDF(html_completo, dest=pdf_file)

if not pisa_status.err:
    print(f"¡PDF generado con éxito!: {ruta_pdf}")
else:
    print("Ocurrió un error al generar el PDF.")

getSize: Not a float '100%'
getSize: Not a float '100%'
getSize: Not a float '100%'
getSize: Not a float '100%'


¡PDF generado con éxito!: reports/Informe_Ejecutivo_Ventas.pdf
